# Andrew Sparkes - DSAI201 Section 01 - Final Project - World Happiness Analysis

# Report Content
The report topic is up to you, and intentionally left open-ended. In loose terms, the report takes some material learned in class and applies it to a problem that interests you. This could be an application to another class, a problem that you conceive, or an application to a hobby or personal interest. You could even merge concepts from different modules in a meaningful way. The point is that you are more likely to use and appreciate Data Science if you actively look for its applications to other areas or interests. So, part of the exercise is coming up with a topic! If you are truly stuck and convinced that you have no idea where to start, schedule a time to brainstorm with me. Otherwise consider these options for project-topic-exploration:

# Report Structure
Your report should roughly follow the structure of a traditional academic report or research paper. This means your final document should contain a title, brief abstract and main body that fully describes your project. An effective main body contains sections such as: (1) introduction; (2) description of technical methods; (3) demonstration of key findings and results; and (4) a summary of the project highlighting the findings, challenges, and possible shortcomings.

The report should provide enough detail that a student at another university could read and fully understand the problem and findings with sufficient effort. This means the introduction should be well-developed and provide the background information necessary to understand the problem. If you are unsure if your message and writing are clear, ask one of your peers to read it. They do NOT have to be an advanced Maths/Data Science student to understand your introduction and main findings. The technical aspects need to be well- developed, well-described, and correct using techniques from class. All figures or tables necessary for presenting results must be created with appropriate software (probably the Matplotlib library). The report must be written using appropriate academic language with correct grammar and coherent sentences. The final document will be a pdf created with LaTeX.

# Report Grading
We will craft a rubric detailing how the reports will be scored. Roughly speaking, you will be assessed on elements of presentation & style, sufficient problem development, technical content, diagrams & illustrations, and overall conclusion. Keep in mind this report and presentation serve as a ‘final’ product, so the stakes are deliberately high with the expectation that you produce quality work. The final report is due Sunday, December 14th.
- Revisit former/current math classes. Are there applications that would benefit from an automated implementation via code?
- Personal/professional interests. Search for data that interests you and apply content to these datasets. You have the freedom to expand application beyond content covered in the course. Just be ready to provide the details in your report!


What are essential components in an effective technical report?

- Shoot for a 150 word abstract.
	* What makes a good abstract?
	* A concise version of a strong introduction.

- An understood expectation of professionalism:
	* Minimal grammar mistakes and typos.
	* Typesetting in LaTeX is read-able.
	* Crediting others for work that is not your own (including images).
	* Proper use of sections & headers. Generally, an organized document.
	* The LaTeX document compiles without any errors. Warnings are OK. Errors are bad.
	* Concise language.


- Effective graphs/figures/tables
	* The graphs match the conclusions/findings and relate to the content of the report.
	* Axes are labeled.
	* Figures with multiple plots/lines/scatters have a legend.
	* All line/scatter sizes are big enough to see, but not so big to be distracting.
	* All figures are numbered, have a caption, and are referenced in the text.
	* Font sizes are big enough to read, but not distracting.

- Conclusion section
	* You did the work. What can you say?
	* Acknowledge the shortcomings of your work.
	* Speculate future directions based on the limitations of your work.

- An effective introduction to the problem/project.
	* Start broad with a few sentences that are understandable to any undergraduate student.
	* Establish current knowledge and a knowledge gap. How will you address it?
	* A clearly stated (probably one sentence) "The goal of this work is to .... "
	* Outline the document structure.

- Maths/Stats/Data
	* Is all correct.
	* Presented in a way that any maths/stats/DS major can read with reasonable effort.
	* Notation should *make sense* following standard conventions.
	* Be careful with re-using letters.


In [ ]:
# setup and imports
import pandas as pd
import numpy as np
import scipy
import kaggle
import kagglehub
import os
import time
import matplotlib.pyplot as plt
import statsmodels.api as sm

# User config settings
alpha = 0.05  # significance level
folder = "plots"  # folder to save plots

LOWESS = True
LINEAR_REGRESSION = True

print(
    f"Using the following settings for calculations:\nalpha = {alpha}\nLOWESS = {LOWESS}\nLINEAR_REGRESSION = {LINEAR_REGRESSION}\nfolder = '{folder}'"
)

print("Setup complete.")


In [ ]:
start_time = time.perf_counter()

# kaggle.api.authenticate() # apparently don't need to authenticate manually, apparently the api call below uses my api key

# pretty sure the dataset_download_files call will make this folder anyway, but just in case...
if not os.path.exists("data"):
    os.makedirs("data")


# download the dataset and import into a df to work with
kaggle.api.dataset_download_files(
    "nalisha/world-happiness-ranking-dataset",
    path="data",
    unzip=True,  # unzip the file after downloading
)

filepath = (
    os.getcwd() + "\\data\\world_happiness_report.csv"
)  # this is the full absolute path
filepath = filepath.replace("\\\\", "\\")
print(f"{filepath=}")

df = pd.read_csv(filepath)
print(df.head())  # sanity check to see if data loaded correctly

print(f"Dataset downloaded and loaded in {time.perf_counter() - start_time}s.")


In [ ]:
# now that we have the data imported, let's ensure it's in a proper form (dataframe) and filter down to the desired years
# well, read_csv already gives us a dataframe, so we just need to filter by year
years_of_interest = [2015, 2016]
df_filtered = df[df["year"].isin(years_of_interest)]
print(df_filtered["year"].unique())  # should only show 2015 and 2016 sanity check

# we're going to be looking at aggregate stats, so no more need to subset or filter the data further


In [ ]:
# define a helper function to determine the strength of the correlation
def correlation_strength(r):
    abs_r = abs(r)
    if abs_r >= 0.75:
        return "Strong"
    elif abs_r >= 0.5:
        return "Moderate"
    elif abs_r >= 0.25:
        return "Weak"
    else:
        return "Statistically Insignificant"


In [ ]:
start_time = time.perf_counter()

# First things first, our basic analysis will be to use numpy to calculate a correlation matrix for the dataset, then we'll extract the r-value and store it for later use

# make a dict for the factors, to hold their r-value later
factors = {
    "Economy (GDP per Capita)": 0,
    "Family": 0,
    "Health (Life Expectancy)": 0,
    "Freedom": 0,
    "Trust (Government Corruption)": 0,
    "Generosity": 0,
    "Dystopia Residual": 0,
}

# calculate r-values
for fac, count in factors.items():
    print(f"Testing correlation between {fac} and Happiness Score...")

    correlation_matrix = np.corrcoef(df_filtered[fac], df_filtered["Happiness Score"])
    r_value = correlation_matrix[0, 1]

    print(
        f"The correlation coefficient (r-value) between {fac} and Happiness Score is: {r_value}, which by our definition is considered a {correlation_strength(r_value)} correlation."
    )
    print("-" * 80)  # divider for readability
    factors[fac] = r_value  # store the r-value for later

factors = dict(
    sorted(factors.items(), key=lambda item: item[1], reverse=True)
)  # sort the factors by their r-value

# output the ranked factors
print("Ranking of factors by correlation with Happiness Score:")
for idx, (k, v) in enumerate(factors.items()):
    print(
        f"Rank {idx + 1}: {k} with r-value of {v} ({correlation_strength(v)} correlation)"
    )

print(f"Done in {time.perf_counter() - start_time}s.")


In [ ]:
start_time = time.perf_counter()
# now that we have our data organized and correlation coefficients calculated, let's visualize the results using some plots

# check for plots folder, create if it doesn't exist
if not os.path.exists(folder):
    os.makedirs(folder)

for idx, (fac, r) in enumerate(factors.items()):
    fig, ax = plt.subplots(nrows=1, ncols=1)  # create figure & 1 axis

    # TODO consider resizing the figure as needed, tbd

    # scatter plot of all the individual data points
    ax.plot(df_filtered[fac], df_filtered["Happiness Score"], "o", label="Data points")

    if LOWESS:
        # now, calculate the LOWESS smoothed line
        smoothed = sm.nonparametric.lowess(
            endog=df_filtered["Happiness Score"], exog=df_filtered[fac], frac=0.2
        )  #

        # The 'smoothed' variable is an array where the first column is x-values and the second is y-values
        smoothed_x = smoothed[:, 0]
        smoothed_y = smoothed[:, 1]

        plt.plot(
            smoothed_x,
            smoothed_y,
            color="red",
            linewidth=3,
            label="LOWESS Smoothing Line",
        )

    if LINEAR_REGRESSION:
        # now, calculate the linear regression line
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(
            df_filtered[fac], df_filtered["Happiness Score"]
        )

        # create x values for the regression line
        x_vals = np.array(ax.get_xlim())
        y_vals = intercept + slope * x_vals  # good ole y = mx+b

        plt.plot(
            x_vals, y_vals, color="orange", linewidth=2, label="Linear Regression Line"
        )

    ax.set_title(
        f"Scatter Plot of {fac} vs Happiness Score\nr-value = {r:.4f} ({correlation_strength(v)} correlation"
    )
    ax.set_xlabel(fac)
    ax.set_ylabel("Happiness Score")
    ax.legend()

    filename = f"{folder}/{idx + 1}_{fac.replace(' ', '_').replace('(', '').replace(')', '')}_scatter_w_regression.png"
    fig.savefig(filename)  # save the figure to file
    plt.close(fig)

    print(f"Saved scatter plot for {fac} as {filename} ...")

print(
    f"Plots generated successfully in {folder} in {time.perf_counter() - start_time}s."
)


In [ ]:
start_time = time.perf_counter()

# performing additional correlation coefficient calculations for comparison, curiosity, and to see how the results compare with numpy's correlation matrix results
factors = {
    "Economy (GDP per Capita)": 0,
    "Family": 0,
    "Health (Life Expectancy)": 0,
    "Freedom": 0,
    "Trust (Government Corruption)": 0,
    "Generosity": 0,
    "Dystopia Residual": 0,
}
for fac, count in factors.items():
    print(
        f"Testing Pearson's R, Spearman's Rho and Kendall's Tau correlation tests between {fac} and Happiness Score..."
    )

    r = scipy.stats.pearsonr(
        df_filtered[fac], df_filtered["Happiness Score"]
    )  # Pearson's R
    rho = scipy.stats.spearmanr(
        df_filtered[fac], df_filtered["Happiness Score"]
    )  # Spearman's Rho
    tau = scipy.stats.kendalltau(
        df_filtered[fac], df_filtered["Happiness Score"]
    )  # Kendall's Tau

    print(
        f"The Pearson's R correlation coefficient between {fac} and Happiness Score is: {r.statistic}, with a p-value of {r.pvalue}, making this a {correlation_strength(r.statistic)} correlation."
    )
    if r.pvalue <= alpha:
        print("(p < alpha) => statistically significant!")
    print(
        f"The Spearman's Rho correlation coefficient between {fac} and Happiness Score is: {rho.statistic}, with a p-value of {rho.pvalue}, making this a {correlation_strength(rho.statistic)} correlation."
    )
    if rho.pvalue <= alpha:
        print("(p < alpha) => statistically significant!")
    print(
        f"The Kendall's Tau correlation coefficient between {fac} and Happiness Score is: {tau.statistic}, with a p-value of {tau.pvalue}, making this a {correlation_strength(tau.statistic)} correlation."
    )
    if tau.pvalue <= alpha:
        print("(p < alpha) => statistically significant!")
    print("-" * 80)
    factors[fac] = r_value  # store the r-value for later

print(
    f"Analysis with Pearson's, Spearman's and Kendall's complete in {time.perf_counter() - start_time}s. These are used only for reference to the numpy correlation matrix results above."
)
